In [ ]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import kagglehub


In [ ]:


!pip install -q \
transformers==4.52.4 \
huggingface_hub==0.33.4 \
accelerate==1.8.1 \
datasets==3.6.0 \
openai-whisper \
librosa \
soundfile \
sentencepiece

print("=" * 70)
print("All required libraries installed successfully!")
print("=" * 70)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.2 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 88.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.3/515.3 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.4 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
diffusers 0.37.1 requires

In [ ]:

import os
import json
import warnings
import numpy as np
import pandas as pd

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

warnings.filterwarnings("ignore")

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("Notebook 3 - Mental Health Assessment Framework")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

print("Device  :", device)

print("=" * 70)

Notebook 3 - Mental Health Assessment Framework
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
Device  : cuda


In [ ]:


import os

print("="*70)
print("Searching Kaggle Input Datasets...")
print("="*70)

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for f in files:
        print("   ", f)

Searching Kaggle Input Datasets...
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/ngay03112993
/kaggle/input/datasets/ngay03112993/distilbert-text-model
    config.json
    tokenizer.json
    tokenizer_config.json
    model.safetensors
    label2id.json
    id2label.json
/kaggle/input/datasets/ngay03112993/voice-model
    voice_analysis_results.json


In [ ]:


import json

RESULTS_PATH = "/kaggle/input/datasets/ngay03112993/voice-model/voice_analysis_results.json"

with open(RESULTS_PATH, "r") as f:
    voice_results = json.load(f)

transcript = voice_results["transcript"]
speech_predictions = voice_results["speech_emotion_predictions"]
text_predictions = voice_results["text_emotion_predictions"]

print("=" * 70)
print("Voice Analysis Results Loaded Successfully!")
print("=" * 70)

print("\nTranscript:")
print(transcript)

print("\nSpeech Predictions:")
print(speech_predictions)

print("\nText Predictions:")
print(text_predictions)

Voice Analysis Results Loaded Successfully!

Transcript:
I had a friend very close to me but recently after like many things have happened so I feel like I again gave her mental pressure she used to be happy I made sure that she won't get to undergo all those stuff that she undergone in her bit it but again I heard her I feel really very bad I don't want to speak to anyone I want to be alone I want to be away from everyone's life I just want to end my life I don't know to say anything

Speech Predictions:
[{'score': 0.6478403806686401, 'label': 'sad'}, {'score': 0.33665764331817627, 'label': 'fearful'}, {'score': 0.007624777499586344, 'label': 'disgust'}, {'score': 0.0051476843655109406, 'label': 'calm'}, {'score': 0.0016167351277545094, 'label': 'happy'}, {'score': 0.0007707081385888159, 'label': 'surprised'}, {'score': 0.0003419941058382392, 'label': 'angry'}]

Text Predictions:
[{'emotion': 'disappointment', 'confidence': 0.9475431442260742}, {'emotion': 'sadness', 'confidence': 0.9

In [ ]:


# Extract highest confidence speech emotion
speech_top = max(speech_predictions, key=lambda x: x["score"])

# Extract highest confidence text emotion
text_top = max(text_predictions, key=lambda x: x["confidence"])

speech_label = speech_top["label"]
speech_conf = speech_top["score"]

text_label = text_top["emotion"]
text_conf = text_top["confidence"]


# Adaptive Confidence-Guided Fusion (ACGF)


if text_conf >= speech_conf:
    final_emotion = text_label
    final_confidence = text_conf
    dominant_modality = "Text"
else:
    final_emotion = speech_label
    final_confidence = speech_conf
    dominant_modality = "Speech"

fusion_result = {
    "speech_emotion": speech_label,
    "speech_confidence": round(speech_conf, 4),

    "text_emotion": text_label,
    "text_confidence": round(text_conf, 4),

    "dominant_modality": dominant_modality,

    "final_emotion": final_emotion,
    "fusion_confidence": round(final_confidence, 4)
}

print("="*70)
print("Adaptive Confidence-Guided Fusion (ACGF)")
print("="*70)

for key, value in fusion_result.items():
    print(f"{key:22s}: {value}")

Adaptive Confidence-Guided Fusion (ACGF)
speech_emotion        : sad
speech_confidence     : 0.6478
text_emotion          : sadness
text_confidence       : 0.9958
dominant_modality     : Text
final_emotion         : sadness
fusion_confidence     : 0.9958


In [ ]:

# Severity scores for emotions
emotion_severity = {

    # High Severity
    "grief": 10,
    "sadness": 9,
    "fear": 8,
    "fearful": 8,
    "remorse": 8,
    "disappointment": 7,
    "disgust": 7,
    "anger": 7,
    "nervousness": 6,

    # Moderate
    "annoyance": 5,
    "confusion": 5,
    "embarrassment": 5,
    "realization": 4,

    # Positive
    "neutral": 3,
    "approval": 2,
    "curiosity": 2,
    "optimism": 2,
    "gratitude": 1,
    "joy": 1,
    "love": 1,
    "relief": 1,
    "admiration": 1,
    "amusement": 1,
    "caring": 1,
    "desire": 1,
    "excitement": 1,
    "pride": 1,
    "surprise": 2,
    "happy": 1,
    "calm": 1
}

severity_score = emotion_severity.get(final_emotion.lower(), 5)

# Mental Health Trajectory Index (0–100)
mhti_score = round((severity_score / 10) * 100, 2)

# Risk Category
if mhti_score >= 80:
    risk = "Critical"

elif mhti_score >= 60:
    risk = "High"

elif mhti_score >= 40:
    risk = "Moderate"

else:
    risk = "Low"

mhti_result = {
    "Final Emotion": final_emotion,
    "Severity Score": severity_score,
    "MHTI Score": mhti_score,
    "Risk Level": risk
}

print("="*70)
print("Mental Health Trajectory Index (MHTI)")
print("="*70)

for k, v in mhti_result.items():
    print(f"{k:20s}: {v}")

Mental Health Trajectory Index (MHTI)
Final Emotion       : sadness
Severity Score      : 9
MHTI Score          : 90.0
Risk Level          : Critical


In [ ]:

# Clinical Priority Assessment (CPA)

if risk == "Critical":
    priority = "Priority 1 - Immediate Psychological Intervention"
    monitoring = "Continuous Monitoring"
    recommendation = (
        "Immediate clinical assessment recommended. "
        "Notify psychologist and initiate crisis intervention if required."
    )

elif risk == "High":
    priority = "Priority 2 - Urgent Follow-up"
    monitoring = "Daily Monitoring"
    recommendation = (
        "Schedule psychologist consultation within 24 hours."
    )

elif risk == "Moderate":
    priority = "Priority 3 - Regular Monitoring"
    monitoring = "Weekly Monitoring"
    recommendation = (
        "Continue journaling and schedule routine counselling sessions."
    )

else:
    priority = "Priority 4 - Routine Monitoring"
    monitoring = "Monthly Monitoring"
    recommendation = (
        "Maintain healthy emotional habits and continue periodic assessments."
    )

cpa_result = {
    "Clinical Priority": priority,
    "Monitoring Frequency": monitoring,
    "Recommendation": recommendation
}

print("=" * 70)
print("Clinical Priority Assessment (CPA)")
print("=" * 70)

for key, value in cpa_result.items():
    print(f"{key:25s}: {value}")

Clinical Priority Assessment (CPA)
Clinical Priority        : Priority 1 - Immediate Psychological Intervention
Monitoring Frequency     : Continuous Monitoring
Recommendation           : Immediate clinical assessment recommended. Notify psychologist and initiate crisis intervention if required.


In [ ]:

print("=" * 80)
print("        AI-BASED MENTAL HEALTH ASSESSMENT REPORT")
print("=" * 80)

print("\nPATIENT JOURNAL")
print("-" * 80)
print(transcript)

print("\nTEXT EMOTION")
print("-" * 80)
print(f"Emotion    : {text_label}")
print(f"Confidence : {text_conf:.4f}")

print("\nSPEECH EMOTION")
print("-" * 80)
print(f"Emotion    : {speech_label}")
print(f"Confidence : {speech_conf:.4f}")

print("\nADAPTIVE CONFIDENCE-GUIDED FUSION (ACGF)")
print("-" * 80)
print(f"Dominant Modality : {dominant_modality}")
print(f"Final Emotion     : {final_emotion}")
print(f"Fusion Confidence : {final_confidence:.4f}")

print("\nMENTAL HEALTH TRAJECTORY INDEX (MHTI)")
print("-" * 80)
print(f"MHTI Score     : {mhti_score}")
print(f"Risk Level     : {risk}")

print("\nCLINICAL PRIORITY ASSESSMENT (CPA)")
print("-" * 80)
print(f"Priority       : {priority}")
print(f"Monitoring     : {monitoring}")

print("\nRECOMMENDATION")
print("-" * 80)
print(recommendation)

print("\n" + "=" * 80)
print("Assessment Completed Successfully")
print("=" * 80)

        AI-BASED MENTAL HEALTH ASSESSMENT REPORT

PATIENT JOURNAL
--------------------------------------------------------------------------------
I had a friend very close to me but recently after like many things have happened so I feel like I again gave her mental pressure she used to be happy I made sure that she won't get to undergo all those stuff that she undergone in her bit it but again I heard her I feel really very bad I don't want to speak to anyone I want to be alone I want to be away from everyone's life I just want to end my life I don't know to say anything

TEXT EMOTION
--------------------------------------------------------------------------------
Emotion    : sadness
Confidence : 0.9958

SPEECH EMOTION
--------------------------------------------------------------------------------
Emotion    : sad
Confidence : 0.6478

ADAPTIVE CONFIDENCE-GUIDED FUSION (ACGF)
--------------------------------------------------------------------------------
Dominant Modality : Text
Fi

In [ ]:

import json

final_report = {
    "Transcript": transcript,

    "Speech Emotion": {
        "Emotion": speech_label,
        "Confidence": round(speech_conf, 4)
    },

    "Text Emotion": {
        "Emotion": text_label,
        "Confidence": round(text_conf, 4)
    },

    "Adaptive Confidence-Guided Fusion (ACGF)": fusion_result,

    "Mental Health Trajectory Index (MHTI)": mhti_result,

    "Clinical Priority Assessment (CPA)": cpa_result
}

output_path = "/kaggle/working/final_mental_health_report.json"

with open(output_path, "w") as file:
    json.dump(final_report, file, indent=4)

print("=" * 70)
print("Final Mental Health Report Saved Successfully!")
print("=" * 70)
print("Saved File:")
print(output_path)

Final Mental Health Report Saved Successfully!
Saved File:
/kaggle/working/final_mental_health_report.json
